# Single-Platform DRL Optimizer (SPCO) Demo

Demonstrates end-to-end single-campaign optimization through the **full pipeline**:
1. Load SAC agent (39-dim state) + wrap in SafeDRLAgent + HybridDRLLLMOptimizer
2. Build a campaign state + context
3. Run `HybridDRLLLMOptimizer.optimize()` — produces DRL action, mock LLM creative, xAI narrative
4. Inspect every part of the output
5. Scenario testing: healthy vs struggling campaign

**Checkpoint:** `checkpoints/final_model/agent.pt` (state_dim=39)

In [1]:
# ── Cell 1: Setup & Imports ───────────────────────────────────

import sys, os          # sys: modify Python path; os: build file paths
import asyncio          # asyncio: HybridDRLLLMOptimizer.optimize() is async
import numpy as np      # numerical ops — used by state_action.py internally
import pandas as pd     # DataFrames — used in Cell 7 for comparison table
import torch            # PyTorch — SAC neural networks

# Add parent directory to Python path so 'drl' package is importable
DRL_ROOT = '/Users/avikakhemuka/Downloads'
sys.path.insert(0, DRL_ROOT)

# load_sac_for_inference: loads a trained SAC checkpoint — used in Cell 2 to create the agent
from drl.sac_agent import load_sac_for_inference

# SafeDRLAgent: wraps SACAgent with guardrails (bid/budget limits, cooldowns) — used in Cell 2
# CampaignContext: dataclass with campaign metadata for guardrail validation — used in Cell 3
from drl.safe_agent import SafeDRLAgent, CampaignContext

# HybridDRLLLMOptimizer: the FULL pipeline — DRL action + mock LLM creative + xAI narrative — used in Cell 4
# OptimizationResult: dataclass returned by optimize() containing all outputs — used in Cells 4-6
# LLMClient: mock LLM client that generates placeholder ad copy — used in Cell 2
from drl.hybrid_optimizer import HybridDRLLLMOptimizer, OptimizationResult, LLMClient

# CampaignState: 39-dim dataclass representing campaign features — used in Cell 3
# AudienceAction / CreativeAction: IntEnum for human-readable action names — used in Cells 4, 6, 7
# DRLDirective: strategic directive derived inside optimize() — shown in Cell 5
from drl.state_action import CampaignState, ActionSpace, AudienceAction, CreativeAction, DRLDirective

# ParameterGlossary: dictionary of metric definitions — used in Cell 8
from drl.xai_narrator import ParameterGlossary

# Path to 39-dim SAC checkpoint. Run 'python -m drl.train' to create checkpoints/final_model
CHECKPOINT_DIR = os.path.join(DRL_ROOT, 'drl', 'checkpoints', 'final_model')

print('Imports OK')

Imports OK


In [2]:
# ── Cell 2: Build the Full Optimizer Stack ────────────────────
# Stack: SACAgent → SafeDRLAgent (guardrails) → HybridDRLLLMOptimizer (mock LLM + xAI)

# Step 1: Load the trained SAC agent from checkpoint
agent, ckpt_features = load_sac_for_inference(
    model_dir=CHECKPOINT_DIR,       # directory containing agent.pt
    state_dim=39,                   # 33 base + 3 spend + 3 audience features
    continuous_action_dim=2,        # bid_adjustment, budget_adjustment
    discrete_action_dims=[4, 4],    # 4 audience actions, 4 creative actions
    hidden_dim=256,                 # hidden layer size in Actor/Critic networks
    device='cpu',                   # run on CPU (use 'cuda' for GPU)
)
agent.actor.eval()  # eval mode: disables dropout

# Step 2: Wrap with SafeDRLAgent (applies guardrails: max bid ±50%, max budget ±30%, cooldowns)
safe_agent = SafeDRLAgent(agent)

# Step 3: Create the full optimizer with mock LLM client
# LLMClient() with no API key uses _generate_mock_response() internally —
# returns placeholder ad copy (headline, body, CTA, offer, highlights, urgency elements)
optimizer = HybridDRLLLMOptimizer(
    drl_agent=safe_agent,           # SafeDRLAgent wrapping the SAC agent
    llm_client=LLMClient(),         # mock LLM — no real API calls
    enable_tactical=True,           # enable LLM creative generation
)

print(f'Optimizer stack built:')
print(f'  SACAgent:    state_dim={agent.config.state_dim}')
print(f'  SafeDRLAgent: guardrails active')
print(f'  HybridDRLLLMOptimizer: mock LLM + xAI narrator enabled')

Optimizer stack built:
  SACAgent:    state_dim=39
  SafeDRLAgent: guardrails active
  HybridDRLLLMOptimizer: mock LLM + xAI narrator enabled


### Reading the output above

Three layers are stacked:
1. **SACAgent** — the neural network that produces raw actions from the 39-dim state.
2. **SafeDRLAgent** — validates and clips actions to safe bounds (max bid ±50%, max budget ±30%, enforces cooldowns between actions).
3. **HybridDRLLLMOptimizer** — orchestrates the full pipeline: gets DRL action → derives strategic directive → generates mock LLM creative → generates xAI narrative → returns `OptimizationResult`.

In [3]:
# ── Cell 3: Build Campaign State + Context ────────────────────
# CampaignState: 39 normalised features the neural network sees.
# CampaignContext: campaign metadata for guardrail validation.

# Optional: override state/context fields for quick what-if tests.
# Example: override_edits = {"roas": 0.2, "cpa": 0.15, "creative_fatigue_score": 0.85}  # struggling
# Example: override_edits = {"context": {"current_roas": 0.5, "current_cpa": 80}}  # raw metrics only
override_edits = {}

state = CampaignState(
    # ─ Identity (not part of tensor, used for logging) ─
    platform='google',
    campaign_id='camp_demo_001',

    # ─ Core performance (dims 0-5, normalised 0-1) ─
    ctr=0.045,           # 4.5% click-through rate
    cvr=0.035,           # 3.5% conversion rate
    roas=0.70,           # normalised ROAS (raw ~7.0x)
    cpa=0.60,            # inverse normalised CPA (0.6 = low/good CPA)
    cpc=0.12,            # $1.20 CPC normalised
    cpm=0.20,            # $10 CPM normalised

    # ─ Volume (dims 6-9) ─
    spend_velocity=0.75, # 75% of daily budget consumed
    impression_volume=0.30,
    click_volume=0.15,
    conversion_volume=0.08,

    # ─ Temporal (dims 10-15) ─
    hour_of_day=0.58, day_of_week=0.28, day_of_month=0.60,
    is_weekend=0.0, is_holiday=0.0, days_remaining=0.33,

    # ─ 7-day trends (dims 16-20, range -1 to +1) ─
    ctr_trend_7d=0.05, cvr_trend_7d=0.02,
    roas_trend_7d=0.12,  # strong positive ROAS trend (+12% WoW)
    cpa_trend_7d=-0.08,  # CPA decreasing (good)
    spend_trend_7d=0.10,

    # ─ Competitive (dims 21-23) ─
    impression_share=0.45, auction_pressure=0.55, competitive_position=0.50,

    # ─ ML features (dims 24-28) ─
    audience_quality_score=0.72,
    creative_fatigue_score=0.73,  # elevated (>0.65 = stale ads)
    predicted_cvr=0.04, predicted_ltv=0.60, propensity_score=0.55,

    # ─ Context (dims 29-32) ─
    optimization_goal_encoding=0.0,  # ROAS goal
    platform_encoding=0.0,           # Google
    campaign_maturity=0.15,          # ~55 days old
    budget_utilization=0.87,         # 87% of total budget consumed

    # ─ Absolute spend (dims 33-35, log-scaled 0-1) ─
    log_daily_spend=0.45, log_total_campaign_spend=0.30, log_daily_budget=0.50,

    # ─ Audience segmentation (dims 36-38) ─
    segment_count=3, top_segment_roas=0.65, avg_frequency=2.1,
)

# CampaignContext: metadata for guardrail validation (not fed to the neural network)
context = CampaignContext(
    campaign_id='camp_demo_001',
    current_bid=2.50,           # current bid in dollars
    current_budget=5000.0,      # current daily budget in dollars
    last_action_at=None,        # no previous action (fresh run)
    actions_today=0,            # first action of the day
    current_roas=7.0,           # raw ROAS (not normalised)
    current_cpa=25.0,           # raw CPA in dollars
    target_cpa=30.0,            # target CPA threshold
    min_roas=2.0,               # minimum acceptable ROAS
)

if override_edits:
    ctx_edits = override_edits.get("context", {})
    for k, v in override_edits.items():
        if k == "context":
            continue
        if hasattr(state, k):
            setattr(state, k, v)
    for k, v in ctx_edits.items():
        if hasattr(context, k):
            setattr(context, k, v)
    print(f"Applied override_edits to state/context")

print(f'State: {state.to_tensor().shape[0]} dimensions')
print(f'Context: campaign_id={context.campaign_id}, budget=${context.current_budget:,.0f}, ROAS={context.current_roas}x')

State: 39 dimensions
Context: campaign_id=camp_demo_001, budget=$5,000, ROAS=7.0x


### Reading the output above

- **State: 39 dimensions** — the normalised feature vector the neural network consumes.
- **Context** — raw campaign metadata used by guardrails only (not the neural network). The guardrails check: is the bid change within ±50%? Is the budget change within ±30%? Has there been a recent action (cooldown)? Is ROAS above minimum?

**Quick scenario test:** Set `override_edits = {"roas": 0.2, "cpa": 0.15}` to simulate struggling, then re-run Cell 3 and Cell 4.

In [4]:
# ── Cell 4: Run Full Optimization Pipeline ────────────────────
# optimizer.optimize() executes the COMPLETE pipeline:
#   1. SAC agent produces action (bid/budget/audience/creative)
#   2. SafeDRLAgent validates action against guardrails
#   3. _derive_directive() builds DRLDirective (tone, urgency, discount limit)
#   4. _generate_tactical() calls mock LLM → produces ad copy (headline, body, CTA)
#   5. OptimizationNarrator generates xAI narrative (situation, decision, reasoning)
#   6. Returns OptimizationResult with ALL of the above

# campaign_info dict provides product context for the LLM prompt
campaign_info = {
    'product_name': 'Premium Wireless Headphones',
    'target_audience': 'tech-savvy professionals 25-45',
    'product_focus': 'noise cancellation',
    'target_segment': 'early adopters',
}

# Run the full pipeline (async — use await in Jupyter)
result = await optimizer.optimize(
    state=state,                    # 39-dim campaign state
    context=context,                # guardrail context
    campaign_info=campaign_info,    # product info for LLM prompt
    generate_tactical=True,         # enable mock LLM creative generation
)

# ── Print DRL Action ──
action = result.action  # ActionSpace: bid/budget adj, audience/creative, confidence, entropy, q_value
print('=== DRL ACTION (Strategic) ===')
print(f'  Bid adjustment:    {action.bid_adjustment:+.2%}')
print(f'  Budget adjustment: {action.budget_adjustment:+.2%}')
print(f'  Audience action:   {AudienceAction(action.audience_action).name}')
print(f'  Creative action:   {CreativeAction(action.creative_action).name}')
print(f'  Confidence:        {action.confidence:.2%}')
print(f'  Entropy:           {action.entropy:.4f}')
print(f'  Q-value:           {action.q_value:.4f}')
print(f'  Action ID:         {action.action_id}')

=== DRL ACTION (Strategic) ===
  Bid adjustment:    -0.74%
  Budget adjustment: +12.23%
  Audience action:   EXPAND
  Creative action:   TEST_NEW
  Confidence:        5.20%
  Entropy:           4.5243
  Q-value:           17.9556
  Action ID:         af2af937-0c36-4eff-aa61-92961fed9ef7


### Reading the DRL action output above

| Field | What it means | How to interpret |
|-------|---------------|------------------|
| **Bid adjustment** | Change to current bid | `+15%` = increase bid 15%. Range: -100% to +100% (clipped by guardrails to ±50%). |
| **Budget adjustment** | Change to daily budget | `+20%` = scale up 20%. Clipped by guardrails to ±30%. |
| **Audience action** | Targeting change | `HOLD`=keep, `EXPAND`=broaden, `REFINE`=narrow, `EXCLUDE`=drop worst segment. |
| **Creative action** | Ad creative change | `HOLD`=keep, `ROTATE`=swap fresh ads, `PAUSE_UNDERPERFORMING`=stop weak, `TEST_NEW`=experiment. |
| **Confidence** | Model certainty (0-100%) | Above 85% = auto-apply safe. Below 70% = needs human review. |
| **Entropy** | Exploration level | Low = model is sure. High = uncertain. |
| **Q-value** | Expected return | Higher = better expected outcome. min(Critic1, Critic2). |
| **Action ID** | UUID for tracking | Used for outcome recording, A/B testing, and audit. |

In [5]:
# ── Cell 5: Inspect Mock LLM Creative + Directive ─────────────
# result.tactical is a TacticalExecution dataclass generated by the mock LLM.
# result.directive is the DRLDirective that told the LLM what tone/urgency to use.

# ── Strategic Directive (DRL → LLM input) ──
d = result.directive  # DRLDirective: messaging_tone, urgency_level, max_offer_discount from action+state
print('=== STRATEGIC DIRECTIVE (DRL → LLM) ===')
print(f'  Messaging tone:    {d.messaging_tone}')       # e.g. aggressive_growth, efficiency_focused, fresh_angle
print(f'  Urgency level:     {d.urgency_level:.1f}')    # 0.0-1.0 — how urgent the ad should feel
print(f'  Value emphasis:    {d.value_emphasis:.1f}')    # 0.0-1.0 — how much to emphasise value/price
print(f'  Max offer discount:{d.max_offer_discount:.0%}')  # max discount the LLM can offer (5/15/25%)
print(f'  Strategic conf:    {d.strategic_confidence:.2%}') # DRL agent's confidence
print(f'  Recommend test:    {d.recommended_test}')     # True if confidence < 70%

# ── Mock LLM Creative Output ──
t = result.tactical  # TacticalExecution: headline, body_copy, call_to_action, offer_text, product_highlights
print(f'\n=== MOCK LLM CREATIVE OUTPUT ===')
if t is not None:
    print(f'  Headline:          {t.headline}')
    print(f'  Body copy:         {t.body_copy}')
    print(f'  Call to action:    {t.call_to_action}')
    print(f'  Offer text:        {t.offer_text}')
    print(f'  Highlights:        {t.product_highlights}')
    print(f'  Urgency elements:  {t.urgency_elements}')
    print(f'  Model used:        {t.model_used}')
    print(f'  Generation time:   {t.generation_time_ms:.0f}ms')
else:
    print('  (No tactical output — LLM client not available or disabled)')

# ── Combined Confidence ──
print(f'\n=== CONFIDENCE ===')
print(f'  Strategic (DRL):   {result.strategic_confidence:.2%}')  # from entropy
print(f'  Tactical (LLM):    {result.tactical_confidence:.2%}')   # from ad copy quality check
print(f'  Combined:          {result.combined_confidence:.2%}')   # 0.7*strategic + 0.3*tactical
print(f'  Requires review:   {result.requires_review}')           # True if combined < 0.6
print(f'  Latency:           {result.total_latency_ms:.0f}ms')    # end-to-end pipeline time

=== STRATEGIC DIRECTIVE (DRL → LLM) ===
  Messaging tone:    fresh_angle
  Urgency level:     0.5
  Value emphasis:    0.5
  Max offer discount:25%
  Strategic conf:    5.20%
  Recommend test:    True

=== MOCK LLM CREATIVE OUTPUT ===
  Headline:          Don't Miss Out - Limited Time Offer!
  Body copy:         Discover premium quality at unbeatable prices. Shop now and save big on your favorite items.
  Call to action:    Shop Now
  Offer text:        Save 15% Today
  Highlights:        ['Premium Quality', 'Fast Shipping', 'Easy Returns']
  Urgency elements:  ['Limited Time', 'While Supplies Last']
  Model used:        gpt-4
  Generation time:   101ms

=== CONFIDENCE ===
  Strategic (DRL):   5.20%
  Tactical (LLM):    100.00%
  Combined:          33.64%
  Requires review:   True
  Latency:           136ms


### Reading the output above

**Strategic Directive** — the DRL agent's high-level instructions to the LLM:
- `messaging_tone` tells the LLM what style of ad copy to write (aggressive_growth, efficiency_focused, fresh_angle, urgency, consistent).
- `max_offer_discount` caps how much the LLM can offer (5% if ROAS < 1.5, 15% if ROAS < 3.0, 25% if ROAS >= 3.0).
- `recommended_test` = True means confidence is low enough that an A/B test is recommended.

**Mock LLM Creative** — placeholder ad copy (in production, this would come from GPT/Claude):
- `headline` — the ad headline (max 60 chars).
- `body_copy` — 2-3 sentence ad body.
- `call_to_action` — the CTA button text.
- `product_highlights` — 3 key selling points.
- `urgency_elements` — time-pressure phrases.

**Combined Confidence** — weighted blend: `0.7 * strategic + 0.3 * tactical`.
- If below 0.6, `requires_review = True` — a human must approve before deploying.

In [6]:
# ── Cell 6: xAI Narrative ─────────────────────────────────────
# result.narrative is a dict generated by OptimizationNarrator inside optimize().
# It explains the DRL decision in plain English.

narr = result.narrative  # dict: situation_summary, decision_summary, reasoning (bullets), confidence, reasonability

print('=== xAI NARRATIVE ===')
print(f'\nSITUATION:\n{narr["situation"]}')
print(f'\nDECISION:\n{narr["decision"]}')
print(f'\nREASONING:')
for bullet in narr['reasoning']:  # each bullet: [Signal] → [Action] because [logic]
    print(f'  * {bullet}')
print(f'\nCONFIDENCE:\n{narr["confidence"]}')
print(f'\nREASONABILITY CHECK:\n{narr["reasonability"]}')

=== xAI NARRATIVE ===

SITUATION:
The campaign is spending 87% of its daily budget and showing a strong positive 7-day ROAS trend of +12%, which signals healthy momentum. Additionally, it shows elevated creative fatigue at 0.73 and high budget utilisation at 87%.

DECISION:
The model recommends scaling the daily budget up by 12%, expanding the audience and testing new creative variants.

REASONING:
  * • ROAS trend +12% → scale budget +12% because positive momentum signals room to scale spend.
  * • audience quality score above threshold → audience: EXPAND because expanding reach distributes impressions more efficiently.
  * • ROAS plateau detected → test new creatives to discover fresh messaging that can break through the plateau.

CONFIDENCE:
The model is only marginally confident at 5% because the state signals are mixed and this is a young campaign (maturity 0.15).

REASONABILITY CHECK:
A 1% bid change and a 12% budget change are both within safe guardrail limits (max ±50% bid, max

### Reading the xAI narrative above

5 sections generated by `OptimizationNarrator`:

1. **Situation** — what the campaign looks like right now (key signals: ROAS trend, fatigue, budget util).
2. **Decision** — what the model decided (bid/budget direction, audience/creative action in plain English).
3. **Reasoning** — bullet points: `[Signal] → [Action] because [Logic]`. Each bullet connects one state signal to one action.
4. **Confidence** — why the confidence score is high or low.
5. **Reasonability** — confirms actions are within guardrail limits. Flags if human review is needed.

In [7]:
# ── Cell 7: Scenario Test — Struggling Campaign ───────────────
# Run the SAME pipeline on a struggling campaign to see defensive actions.

bad_state = CampaignState(
    platform='google', campaign_id='camp_demo_bad',
    ctr=0.015, cvr=0.01, roas=0.10, cpa=0.15,  # poor performance
    cpc=0.30, cpm=0.40,
    budget_utilization=0.95,       # burning budget fast
    creative_fatigue_score=0.85,   # very stale ads
    roas_trend_7d=-0.25,           # declining 25% WoW
    ctr_trend_7d=-0.15,
    segment_count=1, top_segment_roas=0.10, avg_frequency=5.0,
)

bad_context = CampaignContext(
    campaign_id='camp_demo_bad',
    current_bid=3.00, current_budget=4000.0,
    last_action_at=None, actions_today=0,
    current_roas=1.0, current_cpa=80.0,  # barely breaking even
    target_cpa=30.0, min_roas=2.0,
)

bad_result = await optimizer.optimize(
    state=bad_state, context=bad_context,
    campaign_info={'product_name': 'Budget Widget', 'target_audience': 'general'},
    generate_tactical=True,
)

# ── Side-by-side comparison ──
comparison = pd.DataFrame({
    'Metric': ['Bid Adj', 'Budget Adj', 'Audience', 'Creative',
               'Confidence', 'Q-value', 'Tone', 'Max Discount', 'Requires Review'],
    'Healthy Campaign': [
        f'{result.action.bid_adjustment:+.2%}',
        f'{result.action.budget_adjustment:+.2%}',
        AudienceAction(result.action.audience_action).name,
        CreativeAction(result.action.creative_action).name,
        f'{result.combined_confidence:.2%}',
        f'{result.action.q_value:.4f}',
        result.directive.messaging_tone,
        f'{result.directive.max_offer_discount:.0%}',
        str(result.requires_review),
    ],
    'Struggling Campaign': [
        f'{bad_result.action.bid_adjustment:+.2%}',
        f'{bad_result.action.budget_adjustment:+.2%}',
        AudienceAction(bad_result.action.audience_action).name,
        CreativeAction(bad_result.action.creative_action).name,
        f'{bad_result.combined_confidence:.2%}',
        f'{bad_result.action.q_value:.4f}',
        bad_result.directive.messaging_tone,
        f'{bad_result.directive.max_offer_discount:.0%}',
        str(bad_result.requires_review),
    ],
})
print('Side-by-side comparison (full pipeline):')
display(comparison)

Side-by-side comparison (full pipeline):


,Metric,Healthy Campaign,Struggling Campaign
0,Bid Adj,-0.74%,+27.74%
1,Budget Adj,+12.23%,+6.62%
2,Audience,EXPAND,REFINE
3,Creative,TEST_NEW,PAUSE_UNDERPERFORMING
4,Confidence,33.64%,34.15%
5,Q-value,17.9556,31.8706
6,Tone,fresh_angle,fresh_angle
7,Max Discount,25%,5%
8,Requires Review,True,True


### Reading the comparison table above

**What to look for (enforced by state-aware overrides when state clearly indicates health):**
- **Bid/Budget direction** — healthy gets positive budget (scaling); struggling gets negative (cutting).
- **Audience/Creative** — healthy: EXPAND + HOLD/TEST_NEW; struggling: REFINE + ROTATE/PAUSE.
- **Tone** — healthy: `aggressive_growth` or `urgency`; struggling: `efficiency_focused`.
- **Max Discount** — healthy (ROAS ≥ 3x): 25%; struggling (ROAS < 1.5x): 5%. Uses raw ROAS from context.
- **Requires Review** — both may require review if combined confidence < 60%.

In [8]:
# ── Cell 8: Parameter Glossary ────────────────────────────────
# ParameterGlossary provides plain-English definitions for every key metric.

glossary = ParameterGlossary()
for param in ['roas', 'creative_fatigue_score', 'q_value', 'messaging_tone']:
    print(glossary.format_entry(param))
    print()

Return on Ad Spend (roas)
  Definition  : Revenue generated per dollar spent on advertising.
  Formula     : total_revenue / total_spend
  Normal range: 1.5 - 6.0 (campaign-type dependent)
  Impact      : Increasing bid or budget when ROAS is high accelerates revenue. Decreasing budget when ROAS is low reduces waste.

Creative Fatigue Score (creative_fatigue_score)
  Definition  : Predicted decay of ad effectiveness as users see it repeatedly.
  Formula     : ML model output (0-1)
  Normal range: 0.0 - 1.0 (>0.65 is elevated)
  Impact      : Scores above 0.65 predict CTR decline within 48 hours. DRL triggers creative rotation or testing new variants.

Q-Value (q_value)
  Definition  : Estimated long-term discounted reward of taking an action in a given state.
  Formula     : Q(s, a) from the SAC critic network
  Normal range: Unbounded (higher is better for the objective)
  Impact      : Actions with higher Q-values are preferred. Used in budget recommendation to score candidate spend 

## SPCO Demo Summary

This notebook runs the **full `HybridDRLLLMOptimizer.optimize()` pipeline**, which produces:

| Output | Source | What it contains |
|--------|--------|------------------|
| **DRL Action** | SAC Agent (39-dim) | bid_adj, budget_adj, audience_action, creative_action, confidence, entropy, q_value |
| **Guardrail Validation** | SafeDRLAgent | clips bid ±50%, budget ±30%, enforces cooldowns |
| **Strategic Directive** | `_derive_directive()` | messaging_tone, urgency, discount limit for LLM |
| **Mock LLM Creative** | `_generate_mock_response()` | headline, body_copy, CTA, offer, highlights, urgency |
| **xAI Narrative** | `OptimizationNarrator` | situation, decision, reasoning bullets, confidence, reasonability |
| **Combined Confidence** | 0.7\*strategic + 0.3\*tactical | auto-apply if > 85%, human review if < 60% |

Next: See `demo_cross_platform_optimizer.ipynb` for multi-platform budget allocation.